# Lab 05 — A/B Test Analysis — Old vs New Page
**Experimentation Basics Track** · Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute conversion rates by experiment group
2. Run a two-proportion z-test and interpret p-value
3. Build a 95% CI for the conversion difference
4. Estimate MDE at 80% power and make a ship decision

## Datasets (this folder)
- `ab_test.csv` — auto-download from `https://raw.githubusercontent.com/TimileyinSamuel/A-B-Testing-for-E-Commerce-Website/main/ab_test.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-05-ab-test-analysis/lab-05-ab-test-analysis.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-05-ab-test-analysis"
# Hosted manifest (matheshcp/ai_course_content, branch main).
MANIFEST_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-05-ab-test-analysis/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("ab_test.csv", "https://raw.githubusercontent.com/TimileyinSamuel/A-B-Testing-for-E-Commerce-Website/main/ab_test.csv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Experimentation Basics Track: Rates, Z-Test, Ship/No-Ship

> **Scenario:** `ab_test.csv` logs **294,478 sessions** with `con_treat` (control/treatment), `page`, `converted` (0/1). Compute conversion rates, a two-proportion z-test, a 95% CI, and a minimum detectable effect — then **conclude whether to ship**.
>
> **You will learn:** group-by rates, standard error, z-test (manual or scipy), power sketch, peeking pitfall.
> **Time:** ~50 minutes. **Level:** Intermediate. **Needs:** pandas + math/scipy. **Env:** 🟢 Colab only.

### Experiment mental map

| Concept | Formula / code | Meaning |
|---|---|---|
| Conversion rate | `conversions / n` | p̂ per group |
| Pooled SE | `√(p(1-p)(1/n₁+1/n₀))` | null variance for z |
| z-stat | `(p₁ − p₀) / SE` | how extreme is the gap |
| 95% CI (diff) | `(p̂₁−p̂₀) ± 1.96·SE_unpooled` | plausible range for lift |
| MDE @ 80% power | `(1.96+0.84)·SE₀` | smallest gap you can reliably see |

---

### 1. Load data (local first, Colab fallback)

In [ ]:
import math, os
import pandas as pd
from scipy import stats

def load_ab():
    local = "ab_test.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/TimileyinSamuel/A-B-Testing-for-E-Commerce-Website/main/ab_test.csv",
            local,
        )
    return pd.read_csv(local)

df = load_ab()
print(df.shape)          # (294478, 5)
print(df.head(3))
print(df["con_treat"].value_counts())
# control     147202
# treatment   147276
print(df["page"].value_counts())
# old_page / new_page each 147239 (note: page and group are almost aligned —
# a few mismatched rows exist in this dataset; classic QA check)


> Always cross-tab `group × page`. Mismatches mean logging bugs or bot traffic — fix before testing.

In [ ]:
print(pd.crosstab(df["con_treat"], df["page"]))


---

### 2. Conversion rates by group

In [ ]:
rates = df.groupby("con_treat")["converted"].agg(["sum", "count", "mean"])
rates.columns = ["conversions", "n", "rate"]
print(rates.round(6))


Expected:

| group | conversions | n | rate |
|---|---|---|---|
| control | 17723 | 147202 | **0.120399** |
| treatment | 17514 | 147276 | **0.118920** |

Treatment is **lower**, not higher: raw diff ≈ **−0.00148** (−1.23% relative).

---

### 3. Two-proportion z-test

Manual (no scipy required):

In [ ]:
c0, n0 = 17723, 147202   # control
c1, n1 = 17514, 147276   # treatment
p0, p1 = c0 / n0, c1 / n1
p_pool = (c0 + c1) / (n0 + n1)
se_pool = math.sqrt(p_pool * (1 - p_pool) * (1/n0 + 1/n1))
z = (p1 - p0) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"p0={p0:.6f}  p1={p1:.6f}  diff={p1-p0:.6f}")
print(f"z={z:.4f}  p={p_value:.4f}")
# p0=0.120399  p1=0.118920  diff=-0.001480
# z=-1.2369  p=0.2161


Equivalent scipy:

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
# optional: counts = [c1, c0]; nobs = [n1, n0]; z, p = proportions_ztest(counts, nobs)


**Decision at α = 0.05:** p ≈ 0.216 > 0.05 → **fail to reject** H₀. No significant difference.

---

### 4. 95% CI for the difference

In [ ]:
se_unpooled = math.sqrt(p1*(1-p1)/n1 + p0*(1-p0)/n0)
diff = p1 - p0
ci = (diff - 1.96*se_unpooled, diff + 1.96*se_unpooled)
print(f"diff={diff:.6f}  95% CI=({ci[0]:.6f}, {ci[1]:.6f})")
# diff=-0.001480  95% CI=(-0.003824, 0.000865)


CI **contains 0** → ship decision: **do not ship** the new page on this evidence (and the point estimate is slightly worse).

---

### 5. Minimum detectable effect @ 80% power

In [ ]:
# alpha=0.05 two-sided, power=0.80 → z_a + z_b ≈ 1.96 + 0.84
mde = (1.959964 + 0.841621) * se_pool
print(f"MDE absolute: {mde:.6f}  (~{mde/p0*100:.2f}% relative to control)")
# MDE absolute: 0.003351  (~2.78% relative to control)


With these sample sizes you can only reliably detect lifts ≳ 0.34 percentage points.

> **Peeking pitfall:** checking the z-test every hour and stopping when p < 0.05 inflates false positives. Pre-register sample size / duration; use sequential methods if you must peek.

---

## Exercises (do these!)

### Exercise 1 — Conversion by group
Print `conversions`, `n`, and `rate` for control and treatment (6 d.p.).
*Expected: control 17723/147202 = 0.120399 · treatment 17514/147276 = 0.118920.*

<details>
<summary>Hint</summary>

`df.groupby("con_treat")["converted"].agg(["sum","count","mean"])`.
</details>

### Exercise 2 — Z-test + p-value
Run the two-proportion z-test. Report z (4 d.p.) and p (4 d.p.). Significant at α=0.05?
*Expected: z ≈ −1.2369, p ≈ 0.2161 → not significant.*

<details>
<summary>Hint</summary>

Pool proportions for SE under H₀; two-sided p via `2*(1-Φ(|z|))`.
</details>

### Exercise 3 — MDE at 80% power
Using the pooled SE from the null, compute MDE = (1.96+0.84)·SE. Report absolute and % of control rate.
*Expected: ≈ 0.00335 absolute ≈ 2.78% of 0.1204.*

<details>
<summary>Hint</summary>

`mde = (1.96 + 0.84) * se_pool`; relative = `mde / p0`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
rates = df.groupby("con_treat")["converted"].agg(["sum", "count", "mean"])
print(rates)
# control:   17723 / 147202 = 0.120399
# treatment: 17514 / 147276 = 0.118920

# --- Solution 2 ---
c0, n0 = 17723, 147202
c1, n1 = 17514, 147276
p0, p1 = c0/n0, c1/n1
p = (c0+c1)/(n0+n1)
se = math.sqrt(p*(1-p)*(1/n0 + 1/n1))
z = (p1-p0)/se
pv = 2*(1 - stats.norm.cdf(abs(z)))
print(f"z={z:.4f} p={pv:.4f}")   # z=-1.2369 p=0.2161
print("significant?", pv < 0.05)  # False

# --- Solution 3 ---
mde = (1.959964 + 0.841621) * se
print(f"MDE={mde:.6f} ({mde/p0*100:.2f}% of control)")
# MDE=0.003351 (2.78% of control)


### What to learn next
- Chi-square test of independence on the same 2×2 table (should agree with z).
- CUPED / variance reduction; sequential testing (mSPRT).
- Power *before* launch: choose n from business MDE, not after the fact.
- Cheat sheet: rates → z/p → CI → MDE → decision; never peek-and-stop naïvely.

*Files in this folder: `ab_test.csv` (local; raw GitHub URL in `load_ab()`). Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
